In [ ]:
import pickle
import threading
import matlab.engine
import numpy as np
from scipy.stats import norm
from tqdm import tqdm
from pathlib import Path

from UQpy.distributions import JointIndependent, Normal
from UQpy.sampling import LatinHypercubeSampling
from UQpy.sampling.stratified_sampling.latin_hypercube_criteria import DistanceMetric, MaxiMin
from UQpy.surrogates.polynomial_chaos import PolynomialChaosExpansion
from UQpy.surrogates.polynomial_chaos.polynomials.TotalDegreeBasis import TotalDegreeBasis
from UQpy.surrogates.polynomial_chaos.regressions import LeastSquareRegression

from LAR import LeastAngleRegression

In [ ]:
# Matlab calculation

MATLAB_ENGINE = None
MATLAB_ENGINE_LOCK = threading.Lock()


def init_matlab():
    global MATLAB_ENGINE

    if MATLAB_ENGINE is None:
        MATLAB_ENGINE = matlab.engine.start_matlab()
        MATLAB_ENGINE.cd(str(Path.cwd()), nargout=0)


def close_matlab():
    global MATLAB_ENGINE

    if MATLAB_ENGINE is not None:
        MATLAB_ENGINE.quit()
        MATLAB_ENGINE = None


def CBP_SPAN3_Mat(X):
    global MATLAB_ENGINE

    if MATLAB_ENGINE is None:
        raise RuntimeError(
            "MATLAB engine is not initialized. "
            "Call init_matlab() first."
        )

    X = np.asarray(X, dtype=float)

    if X.ndim != 2 or X.shape[1] != 8:
        raise ValueError(
            "X must have shape (n_samples, 8)."
        )

    X_mat = matlab.double(X.tolist())
    results = []

    with MATLAB_ENGINE_LOCK:
        for row in X_mat:
            y_mat = MATLAB_ENGINE.CBP_SPAN3_new(
                row,
                nargout=1,
            )

            y = np.asarray(
                y_mat,
                dtype=float,
            ).reshape(-1)

            results.append(y)

    Y = np.vstack(results)
    return Y[:, :15]

In [ ]:
# Distribution transform function

def normal_to_gumbel(U, mu, beta):
    U = np.asarray(U, dtype=float).reshape(-1)

    probabilities = norm.cdf(U)
    probabilities = np.clip(
        probabilities,
        1.0e-15,
        1.0 - 1.0e-15,
    )

    return mu - beta * np.log(
        -np.log(probabilities)
    )


def normal_to_lognormal(U, mu_ln, sigma_ln):
    U = np.asarray(U, dtype=float).reshape(-1)

    return np.exp(
        mu_ln + sigma_ln * U
    )


def predict_all_outputs(pce_models, X_test):
    X_test = np.asarray(X_test, dtype=float)
    columns = []

    for model in pce_models:
        if model is None:
            prediction = np.full(
                X_test.shape[0],
                np.nan,
            )
        else:
            prediction = np.asarray(
                model.predict(X_test),
                dtype=float,
            ).reshape(-1)

        columns.append(prediction)

    return np.column_stack(columns)


def lhs_samples_independent(
    n_samples,
    distributions,
    rng,
    criterion,
):
    seed = int(
        rng.integers(0, 2**32 - 1)
    )

    kwargs = {
        "distributions": distributions,
        "criterion": criterion,
        "nsamples": int(n_samples),
    }

    try:
        lhs = LatinHypercubeSampling(
            **kwargs,
            random_state=seed,
        )
    except TypeError:
        lhs = LatinHypercubeSampling(
            **kwargs,
            seed=seed,
        )

    samples = getattr(lhs, "_samples", None)

    if samples is None:
        samples = getattr(lhs, "samples", None)

    if samples is None:
        raise AttributeError(
            "Could not obtain samples from "
            "LatinHypercubeSampling."
        )

    return np.asarray(samples, dtype=float)

In [ ]:
# PCE and input setting (physical and surrogate space)

# 1-3
D_mean = np.array([30.0, 10.0, 30.0])
D_std = 0.10 * D_mean

dist1 = Normal(loc=D_mean[0], scale=D_std[0])
dist2 = Normal(loc=D_mean[1], scale=D_std[1])
dist3 = Normal(loc=D_mean[2], scale=D_std[2])


# 4-6
L_mean = np.array([20.0, 10.0, 20.0])
L_std = 0.25 * L_mean

beta_L = (np.sqrt(6.0) / np.pi) * L_std
mu_L = L_mean - 0.5772 * beta_L

dist4 = Normal(0, 1)
dist5 = Normal(0, 1)
dist6 = Normal(0, 1)


# 7
m7_mean = 26.7
cov7 = 0.15

sigma_ln7 = np.sqrt(np.log(1.0 + cov7 ** 2))
mu_ln7 = np.log(m7_mean) - 0.5 * sigma_ln7 ** 2

dist7 = Normal(0, 1)


# 8
m8 = 435.0
s8 = 0.105 * m8

dist8 = Normal(m8, s8)


# Joint distribution
DISTS = [dist1, dist2, dist3, dist4, dist5, dist6, dist7, dist8]

joint = JointIndependent(DISTS)


# PCE basis
P = 8
polynomial_basis = TotalDegreeBasis(
    joint,
    P,
    hyperbolic=0.65
)


# Testing set and candidate pool
xx_testing = np.load("xx_testing.npy")
yy_testing = np.load("yy_testing.npy")
Xaptive = np.load("Xaptive.npy")

In [ ]:
# LHS

def run(
    n_repeats=20,
    cbp_func=None,
    init_n=50,
    max_n=500,
    step_size=10,
    nout=15,
    n_test=3000,
    n_var_ref=100000,
    master_seed=2026,
    test_seed=123,
    var_seed_offset=999,
    show_progress=True,
    store_float32=True,
):
    if cbp_func is None:
        cbp_func = CBP_SPAN3_Mat

    nout = int(nout)

    if nout > yy_testing.shape[1]:
        raise ValueError(
            "nout exceeds the number of available outputs."
        )

    # Fixed test subset
    finite_test_mask = np.all(
        np.isfinite(yy_testing[:, :nout]),
        axis=1,
    )

    finite_test_indices = np.flatnonzero(
        finite_test_mask
    )

    if n_test > finite_test_indices.size:
        raise ValueError(
            "n_test exceeds the number of finite testing samples."
        )

    rng_test = np.random.default_rng(
        int(test_seed)
    )

    idx_test = rng_test.choice(
        finite_test_indices,
        size=int(n_test),
        replace=False,
    )

    X_test_sub = xx_testing[idx_test]

    # Reference variance.
    yy_finite = yy_testing[
        finite_test_mask,
        :nout,
    ]

    if yy_finite.shape[0] < 2:
        raise ValueError(
            "At least two finite output samples are required."
        )

    rng_var = np.random.default_rng(
        int(test_seed) + int(var_seed_offset)
    )

    n_var = min(
        int(n_var_ref),
        yy_finite.shape[0],
    )

    idx_var = rng_var.choice(
        yy_finite.shape[0],
        size=n_var,
        replace=False,
    )

    var_true = np.var(
        yy_finite[idx_var],
        axis=0,
        ddof=1,
    )

    rng_master = np.random.default_rng(
        int(master_seed)
    )

    n_train_list = np.arange(
        int(init_n),
        int(max_n) + 1,
        int(step_size),
    )

    lhs_criterion = MaxiMin(
        metric=DistanceMetric.CHEBYSHEV
    )

    results_all = []

    repeat_iterator = range(
        int(n_repeats)
    )

    if show_progress:
        repeat_iterator = tqdm(
            repeat_iterator,
            desc="Repeats",
            leave=True,
        )

    for repeat in repeat_iterator:
        hist_eval = []
        y_pred_hist = []
        var_pce_hist = []
        eps_hist = []
        loo_hist = []

        training_iterator = n_train_list

        if show_progress:
            training_iterator = tqdm(
                training_iterator,
                desc=f"LHS run {repeat + 1}",
                leave=False,
            )

        for n_train in training_iterator:
            X_train = lhs_samples_independent(
                n_samples=n_train,
                distributions=DISTS,
                rng=rng_master,
                criterion=lhs_criterion,
            )

            X_train_phys = X_train.copy()

            for i in range(3):
                column = 3 + i

                X_train_phys[:, column] = normal_to_gumbel(
                    X_train[:, column],
                    mu_L[i],
                    beta_L[i],
                )

            X_train_phys[:, 6] = normal_to_lognormal(
                X_train[:, 6],
                mu_ln7,
                sigma_ln7,
            )

            Y_train = np.asarray(
                cbp_func(X_train_phys),
                dtype=float,
            )

            if Y_train.ndim != 2:
                raise ValueError(
                    "The MATLAB model must return a two-dimensional array."
                )

            if Y_train.shape[1] < nout:
                raise ValueError(
                    "The MATLAB model returned fewer outputs than requested by nout."
                )

            Y_train = Y_train[:, :nout]

            finite_rows = np.all(
                np.isfinite(Y_train),
                axis=1,
            )

            X_train_clean = X_train[finite_rows]
            Y_train_clean = Y_train[finite_rows]

            if X_train_clean.shape[0] < 2:
                continue

            pce_models = []
            loo_error = np.full(
                nout,
                np.nan,
                dtype=float,
            )

            var_pce = np.full(
                nout,
                np.nan,
                dtype=float,
            )

            for j in range(nout):
                model = PolynomialChaosExpansion(
                    polynomial_basis=polynomial_basis,
                    regression_method=LeastSquareRegression(),
                )

                model.fit(
                    X_train_clean,
                    Y_train_clean[:, j],
                )

                try:
                    selected_model = (
                        LeastAngleRegression.model_selection(
                            model
                        )
                    )
                except Exception:
                    selected_model = model

                pce_models.append(
                    selected_model
                )

                try:
                    loo_error[j] = float(
                        selected_model.leaveoneout_error()
                    )
                except Exception:
                    loo_error[j] = np.nan

                try:
                    _, variance = (
                        selected_model.get_moments()
                    )

                    var_pce[j] = float(variance)
                except Exception:
                    var_pce[j] = np.nan

            relative_variance_error = (
                np.abs(var_pce - var_true)
                / np.maximum(var_true, 1.0e-15)
            )

            Y_pred = predict_all_outputs(
                pce_models,
                X_test_sub,
            )

            if store_float32:
                Y_pred = Y_pred.astype(np.float32)
                var_pce = var_pce.astype(np.float32)
                relative_variance_error = (
                    relative_variance_error.astype(np.float32)
                )
                loo_error = loo_error.astype(np.float32)

            hist_eval.append(
                int(n_train)
            )

            y_pred_hist.append(
                Y_pred
            )

            var_pce_hist.append(
                var_pce
            )

            eps_hist.append(
                relative_variance_error
            )

            loo_hist.append(
                loo_error
            )

        output_dtype = (
            np.float32
            if store_float32
            else float
        )

        if y_pred_hist:
            Y_pred_hist_array = np.stack(
                y_pred_hist,
                axis=0,
            )
        else:
            Y_pred_hist_array = np.empty(
                (0, int(n_test), nout),
                dtype=output_dtype,
            )

        results_all.append({
            "run": repeat + 1,
            "hist_eval": np.asarray(
                hist_eval,
                dtype=int,
            ),
            "idx_test": np.asarray(
                idx_test,
                dtype=int,
            ),
            "idx_var_ref": np.asarray(
                idx_var,
                dtype=int,
            ),
            "Y_pred_hist": np.asarray(
                Y_pred_hist_array,
                dtype=output_dtype,
            ),
            "var_pce_hist": np.asarray(
                var_pce_hist,
                dtype=output_dtype,
            ),
            "eps_hist": np.asarray(
                eps_hist,
                dtype=output_dtype,
            ),
            "one_minus_Q2_hist": np.asarray(
                loo_hist,
                dtype=output_dtype,
            ),
            "var_true": np.asarray(
                var_true,
                dtype=output_dtype,
            ),
            "settings": {
                "init_n": int(init_n),
                "max_n": int(max_n),
                "step_size": int(step_size),
                "nout": nout,
                "n_test": int(n_test),
                "n_var_ref_requested": int(n_var_ref),
                "n_var_ref_used": int(n_var),
                "master_seed": int(master_seed),
                "test_seed": int(test_seed),
                "var_seed_offset": int(var_seed_offset),
                "store_float32": bool(store_float32),
                "sampling_method": "independent LHS",
                "var_true_source": (
                    "finite yy_testing rows, "
                    "random subset, ddof=1"
                ),
            },
        })

    return results_all

In [ ]:
init_matlab()

try:
    results = run(
        n_repeats=50,
        cbp_func=CBP_SPAN3_Mat,
        init_n=50,
        max_n=500,
        step_size=10,
        nout=15,
        n_test=3000,
        n_var_ref=100000,
        master_seed=2026,
        test_seed=123,
        show_progress=True,
        store_float32=True,
    )
finally:
    close_matlab()

with open("LHS.pkl", "wb") as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

print("done, total runs:", len(results))